# 统一公司年度研究面板

财务面板是 240 个 firm-year 的骨架；Profile 按公司匹配，Patent 按 firm-year 左连接。专利缺口保留 missing，并用 merge provenance 构造 `patent_record_present`。

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')
from src.build_research_panel import load_processed_panel

processed_dir = Path('../data/processed')
panel = load_processed_panel(processed_dir)
panel.head()

,stock_code,company_name,year,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,...,profile_company_name,province,city,industry,ownership,listing_date,invention_patents,utility_patents,patent_citations,patent_record_present
0,000001,华辰科技股份有限公司,2020,44813001602.0,23908847993.0,8788852319.0,823332467.0,4322096458.0,655221236.0,0.0394,...,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01,8,19,33,1
1,000001,华辰科技股份有限公司,2021,51047456089.0,30247752226.0,12258996387.0,1406421903.0,12041496671.0,680417512.0,0.0676,...,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01,8,16,23,1
2,000001,华辰科技有限公司,2022,53781621074.0,40472674936.0,11646604275.0,-90319782.0,8762245930.0,883046862.0,-0.0068,...,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01,<NA>,<NA>,<NA>,0
3,000001,华辰科技股份有限公司,2023,1934677743.0,476851735.0,17627268570.0,-908582885.0,273990284.0,393474856.0,-0.6232,...,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01,8,9,24,1
4,000001,ST华辰科技股份有限公司,2024,48924685296.0,37663675836.0,35182016317.0,2123243663.0,13679245844.0,2724202038.0,0.1885,...,华辰科技有限公司,广东省,广州市,软件和信息技术服务业,国有,2005-01-01,10,8,29,1


In [2]:
panel.info()
panel.shape
panel.groupby('year').size()
panel['patent_record_present'].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   stock_code             240 non-null    string        
 1   company_name           240 non-null    string        
 2   year                   240 non-null    Int64         
 3   total_assets           239 non-null    Float64       
 4   total_liabilities      239 non-null    Float64       
 5   revenue                240 non-null    Float64       
 6   net_profit             238 non-null    Float64       
 7   cash                   239 non-null    Float64       
 8   rd_expense             239 non-null    Float64       
 9   roe                    240 non-null    Float64       
 10  employees              238 non-null    Int64         
 11  profile_company_name   240 non-null    string        
 12  province               240 non-null    string        
 13  city            

patent_record_present
1    220
0     20
Name: count, dtype: Int64

In [3]:
assert panel.shape[0] == 240
assert panel['stock_code'].nunique() == 40
assert panel['year'].nunique() == 6
assert panel['year'].between(2020, 2025).all()
assert not panel.duplicated(['stock_code', 'year']).any()
assert panel['patent_record_present'].value_counts().to_dict() == {1: 220, 0: 20}
patent_columns = ['invention_patents', 'utility_patents', 'patent_citations']
missing_patents = panel.loc[panel['patent_record_present'] == 0, patent_columns]
assert missing_patents.isna().all().all()
panel.to_parquet(processed_dir / 'research_panel_base.parquet', index=False)
stata = panel.copy()
stata_columns = [
    'year', 'invention_patents', 'utility_patents',
    'patent_citations', 'patent_record_present',
]
for column in stata_columns:
    stata[column] = stata[column].astype(float)
stata.to_stata(processed_dir / 'research_panel_base.dta', write_index=False, version=118)
print('Financial + Profile + Patent panel written successfully.')

Financial + Profile + Patent panel written successfully.


## 合并解释

Financial × Profile 是 financial 左表对 Profile 的 many-to-one 合并，Profile 中的 900001、900002 不进入研究样本。随后按 `stock_code + year` 对 Patent 做 one-to-one 左连接。`patent_record_present` 直接来自连接结果；因此“记录存在但某个专利字段缺失”和“没有专利记录”被明确区分。